# Анализ экспериментов по подстройке коэффициентов Kp, Ki, Kd с включенной термостабилизацией от 13 января 2026 г


In [ ]:
import matplotlib.pyplot as plt

from pathlib import Path
from nn_laser_stabilizer.config.config import load_config

EXPERIMENT_DIR_PATH = Path(f"../experiments/2026-01-12_19-41-32_pid_delta_tuning")

CONFIG_PATH = EXPERIMENT_DIR_PATH / "config.yaml"
ENV_LOG_PATH = EXPERIMENT_DIR_PATH / "env_logs" / "env.log"
TRAIN_LOG_PATH = EXPERIMENT_DIR_PATH / "train_logs" / "train.log"

config = load_config(CONFIG_PATH)
print(f"Эксперимент: {config.experiment_name}")
print(f"Seed: {config.seed}")

## Анализ логов окружения


In [ ]:
import numpy as np
import pandas as pd

from nn_laser_stabilizer.analysis.sources import parse_keyval_log

# env.log этого поколения содержит и строки окружения ([ENV] step/reset),
# и обмен с PID-контроллером ([PID] SEND/READ). parse_keyval_log разбирает
# их в один DataFrame с колонками tag/event; ниже разделяем на env_df и
# connection_df, воспроизводя прежнюю логику.
_raw = parse_keyval_log(ENV_LOG_PATH)

# --- окружение ([ENV]); шаги содержат should_reset, reset — нет ---
_env = _raw[_raw["tag"] == "ENV"].reset_index(drop=True)
_is_step = _env["should_reset"].notna()
env_df = pd.DataFrame({
    "Step": _env["step"],
    "time": _env["time"],
    "Type": np.where(_is_step, "step", "reset"),
    "Kp": _env["kp"],
    "Ki": _env["ki"],
    "Kd": _env["kd"],
    "Delta Kp": _env["delta_kp_norm"].where(_is_step),
    "Delta Ki": _env["delta_ki_norm"].where(_is_step),
    "Delta Kd": _env["delta_kd_norm"].where(_is_step),
    "Error mean norm": _env["error_mean_norm"],
    "Error std norm": _env["error_std_norm"],
    "Reward": _env["reward"].where(_is_step),
    "Should reset": _env["should_reset"].where(_is_step, other=True).astype(bool),
})
env_df["Step"] = env_df["Step"].ffill().fillna(0).astype(int)

# --- соединение ([PID] SEND/READ); i-я посылка спаривается с i-м чтением ---
_pid = _raw[_raw["tag"] == "PID"]
_send = _pid[_pid["event"] == "SEND"][
    ["kp", "ki", "kd", "control_min", "control_max"]
].reset_index(drop=True)
_send["_i"] = range(len(_send))
_read = _pid[_pid["event"] == "READ"][
    ["process_variable", "control_output"]
].reset_index(drop=True)
_read["_i"] = range(len(_read))
_pairs = _read.merge(_send, on="_i", how="left")
connection_df = pd.DataFrame({
    "Connection step": _pairs["_i"],
    "Type": "exchange",
    "Kp": _pairs["kp"],
    "Ki": _pairs["ki"],
    "Kd": _pairs["kd"],
    "Control min": _pairs["control_min"],
    "Control max": _pairs["control_max"],
    "Process variable": _pairs["process_variable"],
    "Control output": _pairs["control_output"],
})

print(f"Загружено {len(env_df)} записей из логов окружения")
print(f"Диапазон шагов: {env_df['Step'].min()} - {env_df['Step'].max()}")

In [ ]:
step_df = env_df[env_df['Type'] == 'step'].copy()
print("=== Статистика по шагам окружения ===")
print(step_df.describe())


In [ ]:
exploration_steps = config.training.exploration_steps
initial_collect_steps = config.training.initial_collect_steps
neural_network_step = max(initial_collect_steps, exploration_steps)

columns_to_plot = ['Error mean norm', 'Error std norm', 'Reward']

for col in columns_to_plot:
    plt.figure(figsize=(12, 5))
    plt.plot(step_df['Step'], step_df[col], alpha=0.8, linewidth=0.8, label=col)
    
    if neural_network_step <= step_df['Step'].max():
        plt.axvline(x=neural_network_step, color='red', linestyle='--', linewidth=2, 
                    label=f'Switch to NN (step {neural_network_step})')
    
    plt.title(f'{col} over Steps')
    plt.xlabel('Step')
    plt.ylabel(col)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

In [ ]:
cols = ['Kp', 'Ki', 'Kd']
fig, axes = plt.subplots(len(cols), 1, figsize=(14, 10), sharex=True)

for ax, col in zip(axes, cols):
    ax.plot(step_df['Step'], step_df[col], alpha=0.8, linewidth=0.8, label=col)

    if neural_network_step <= step_df['Step'].max():
        ax.axvline(
            x=neural_network_step,
            color='red',
            linestyle='--',
            linewidth=2,
            label=f'Switch NN ({neural_network_step})'
        )

    ax.set_ylabel(col)
    ax.grid(True, alpha=0.3)
    ax.legend()

axes[-1].set_xlabel("Step")
plt.suptitle("Kp / Ki / Kd over Steps", fontsize=14)
plt.tight_layout()
plt.show()

## Анализ процесса обучения


In [ ]:
from nn_laser_stabilizer.analysis.sources import parse_train_log

# Новый формат: step time loss_q1 loss_q2 [actor_loss] buffer_size.
loss_df = parse_train_log(TRAIN_LOG_PATH)
loss_df = (
    loss_df[loss_df[["loss_q1", "loss_q2", "buffer_size"]].notna().all(axis=1)]
    .reindex(columns=["step", "loss_q1", "loss_q2", "actor_loss", "buffer_size"])
    .reset_index(drop=True)
)
print(f"Загружено {len(loss_df)} записей из логов обучения")
print(f"Диапазон шагов обучения: {loss_df['step'].min()} - {loss_df['step'].max()}")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

axes[0].plot(loss_df['step'], loss_df['loss_q1'], 'b-', alpha=0.7, label='Q1 Loss')
axes[0].set_title('Q1 Loss')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(loss_df['step'], loss_df['loss_q2'], 'g-', alpha=0.7, label='Q2 Loss')
axes[1].set_title('Q2 Loss')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(loss_df['step'], loss_df['loss_q1'] + loss_df['loss_q2'], 'r--', alpha=0.7, label='Sum (Q1 + Q2)')
axes[2].set_title('Sum (Q1 + Q2)')
axes[2].set_xlabel('Step')
axes[2].set_ylabel('Loss')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))
actor_loss_df = loss_df[loss_df['actor_loss'].notna()]
if len(actor_loss_df) > 0:
    plt.plot(actor_loss_df['step'], actor_loss_df['actor_loss'], 'r-', alpha=0.7)
    plt.title('Actor Loss')
else:
    plt.text(0.5, 0.5, 'No actor loss data', ha='center', va='center', transform=plt.gca().transAxes)
    plt.title('Actor Loss (no data)')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(loss_df['step'], loss_df['buffer_size'], 'm-', alpha=0.7)
plt.title('Buffer Size')
plt.xlabel('Step')
plt.ylabel('Size')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Анализ состояния установки


In [ ]:
setpoint = config.env.args.setpoint
block_size = config.env.args.block_size

neural_network_connection_step = neural_network_step * block_size
plt.figure(figsize=(12, 5))
plt.plot(connection_df['Connection step'], connection_df['Process variable'], 'b-', alpha=0.7, linewidth=0.8, label='Process Variable')
plt.axhline(y=setpoint, color='r', linestyle='--', label=f'Setpoint ({setpoint})')

if neural_network_connection_step <= connection_df['Connection step'].max():
    plt.axvline(x=neural_network_connection_step, color='red', linestyle='--', linewidth=2, 
                label=f'Switch to NN (step {neural_network_connection_step})')

plt.title('Process Variable')
plt.xlabel('Step')
plt.ylabel('Process Variable')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(connection_df['Connection step'], connection_df['Control output'], 'g-', alpha=0.7, linewidth=0.8, label='Control Output')

if neural_network_connection_step <= connection_df['Connection step'].max():
    plt.axvline(x=neural_network_connection_step, color='red', linestyle='--', linewidth=2, 
                label=f'Switch to NN (step {neural_network_connection_step})')

plt.title('Control Output')
plt.xlabel('Step')
plt.ylabel('Control Output')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
env_df = env_df.sort_values('time').copy()

env_df['time_diff'] = env_df['time'].diff()
env_df['time_diff_ms'] = env_df['time_diff'] * 1000  

step_time_df = env_df[env_df['Type'] == 'step'].copy()
time_df = env_df.copy()

print("=== Статистика по времени ===")
print(f"Всего записей: {len(time_df)}")
print(f"Шагов: {len(time_df[time_df['Type'] == 'step'])}")
print(f"Reset событий: {len(time_df[time_df['Type'] == 'reset'])}")

if len(step_time_df) > 0:
    print(f"\n=== Статистика интервалов между шагами ===")
    print(step_time_df['time_diff_ms'].describe())
    print(f"\nМедианный интервал: {step_time_df['time_diff_ms'].median():.2f} мс")
    print(f"Средний интервал: {step_time_df['time_diff_ms'].mean():.2f} мс")
    print(f"Максимальный интервал: {step_time_df['time_diff_ms'].max():.2f} мс")
    print(f"Минимальный интервал: {step_time_df['time_diff_ms'].min():.2f} мс")
    
    plt.figure(figsize=(12, 5))
    plt.plot(step_time_df['Step'], step_time_df['time_diff_ms'], 'b-', alpha=0.7, linewidth=0.8)
    plt.title('Time Intervals Between Steps')
    plt.xlabel('Step')
    plt.ylabel('Time Interval (ms)')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    plt.figure(figsize=(12, 5))
    plt.hist(step_time_df['time_diff_ms'].dropna(), bins=50, alpha=0.7, edgecolor='black')
    plt.title('Distribution of Time Intervals Between Steps')
    plt.xlabel('Time Interval (ms)')
    plt.ylabel('Frequency')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
reset_time_df = env_df[env_df['Type'] == 'reset'].copy()
if len(reset_time_df) > 0:
    print(f"\n=== Статистика интервалов между шагами ===")
    print(reset_time_df['time_diff_ms'].describe())
    print(f"\nМедианный интервал: {reset_time_df['time_diff_ms'].median():.2f} мс")
    print(f"Средний интервал: {reset_time_df['time_diff_ms'].mean():.2f} мс")
    print(f"Максимальный интервал: {reset_time_df['time_diff_ms'].max():.2f} мс")
    print(f"Минимальный интервал: {reset_time_df['time_diff_ms'].min():.2f} мс")
    
    plt.figure(figsize=(12, 5))
    plt.plot(reset_time_df['Step'], reset_time_df['time_diff_ms'], 'b-', alpha=0.7, linewidth=0.8)
    plt.title('Time Intervals Between Steps')
    plt.xlabel('Step')
    plt.ylabel('Time Interval (ms)')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    plt.figure(figsize=(12, 5))
    plt.hist(reset_time_df['time_diff_ms'].dropna(), bins=50, alpha=0.7, edgecolor='black')
    plt.title('Distribution of Time Intervals Between Steps')
    plt.xlabel('Time Interval (ms)')
    plt.ylabel('Frequency')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()